In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import datetime
import eumdac
import os
import requests
import shutil

from IPython.core.display import HTML

In [3]:
consumer_key = os.getenv('CONSUMER_KEY_EUMETSAT')
consumer_secret = os.getenv('CONSUMER_SECRET_EUMETSAT')

if not consumer_secret or not consumer_key:
    raise ValueError("Please set the environment variables CONSUMER_KEY_EUMETSAT and CONSUMER_SECRET_EUMETSAT to your EUMETSAT API credentials.")

credentials = (consumer_key, consumer_secret)

token = eumdac.AccessToken(credentials)
try:
    print(f"This token '{token}' expires {token.expiration}")
except requests.exceptions.HTTPError as error:
    print(f"Error when tryng the request to the server: '{error}'")

This token 'b40d4233-66ee-3c9e-991d-629540473b25' expires 2026-10-01 12:01:25.490173


In [4]:
# Retrieve how many collection objects are available from DataStore
datastore = eumdac.DataStore(token)
try:
    print(f"EUMETSATs DataStore provides access to {len(datastore.collections)} collections.")
except eumdac.datastore.DataStoreError as error:
    print(f"Error related to the data store: '{error.msg}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

EUMETSATs DataStore provides access to 188 collections.


In [5]:
try:
    for collection in datastore.collections:
        print(f"{collection} - {collection.title}")
except eumdac.datastore.DataStoreError as error:
    # The message will be: Load collections failed.
    print(f"Error related to the DataStore:'{error.msg}'")
except eumdac.collection.CollectionError as error:
    # The message will be: Related to a failed loading of properties related to the collection ID.
    print(f"Error related to a collection: '{error.msg}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

EO:EUM:DAT:0959 - IASI Carbon Monoxide Profiles FORLI-CO Climate Data Record Release 1 - Metop-A and -B
EO:EUM:DAT:0960 - IASI SO2 Climate Data Record Release 1 - Metop-A and -B
EO:EUM:DAT:0962 - SEVIRI Rapid Scan High Rate Level 1.5 Image Data Climate Data Record Release 1 - MSG
EO:EUM:DAT:0961 - HIRS Level 1C Fundamental Data Record Release 2 - Multimission - Global
EO:EUM:DAT:0964 - SSM/T Level 1C Fundamental Data Record Release 1 - Multimission - Global
EO:EUM:DAT:0963 - SI-1 Fundamental Data Record Release 1 - Meteor - Polar
EO:EUM:DAT:0737 - IRS Level 1 reduced sub-set of 300 Principal Components - MTG
EO:EUM:DAT:0741 - IRS Level 1 spectral soundings - MTG
EO:EUM:DAT:0986 - Deep Convective Cloud System Monitoring Database - TOOCAN v2.08 Africa
EO:EUM:DAT:0987 - Deep Convective Cloud System Monitoring Database - CACATOES v1.04 Africa
EO:EUM:DAT:0910 - CM SAF Passive Microwave Upper Tropospheric Humidity (UTH) Data Record - Edition 2
EO:EUM:DAT:0908 - Auxiliary data TLE – AWS PFM
E

In [6]:
# Select our collection
selected_collection = datastore.get_collection('EO:EUM:DAT:0662')

# Display the details for the selected collection
try:
    display(HTML('<b>'+selected_collection.title+'</b>'))
    display(HTML('<b>ID:</b> '+str(selected_collection)))
    display(HTML('<b>Abstract:</b> '+selected_collection.abstract))
except eumdac.collection.CollectionError as error:
    print(f"Collection error: {error}")
    #The message will be: Could not get properties of collection id
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: '{error}'")

EO:EUM:DAT:0662 is the MTG FCI Level-1c Normal Resolution collection: calibrated, geolocated satellite radiances (the "images") from the Flexible Combined Imager on MTG-I. It's the input to your model, not a derived product like a cloud mask.


What one product contains:
- Full-disk scans repeat every 10 minutes, so you get 144 time steps per day.
eumetsat
- The instrument has 16 channels: 8 solar channels (0.4-2.2 µm) at 1 km, and 8 thermal channels (3.8-13.3 µm) at 2 km.


In [7]:
try:
    display(selected_collection.search_options)
except eumdac.datastore.DataStoreError as error:
    print(f"Error related to the data store: '{error.msg}'")
except eumdac.collection.CollectionError as error:
    print(f"Error related to the collection: '{error.msg}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

{'bbox': {'title': 'Inventory which has a spatial extent overlapping this bounding box',
  'options': []},
 'geo': {'title': 'Inventory which has a spatial extent overlapping this Well Known Text geometry',
  'options': []},
 'title': {'title': 'Can be used to define a wildcard search on the product title (product identifier), use set notation as OR and space as AND operator between multiple search terms',
  'options': [None]},
 'sat': {'title': 'Mission / Satellite', 'options': ['MTI1']},
 'type': {'title': 'Product Type', 'options': ['MTIFCI1CRRADFDHSI']},
 'dtstart': {'title': 'Temporal Start', 'options': []},
 'dtend': {'title': 'Temporal End', 'options': []},
 'publication': {'title': 'publication date', 'options': []},
 'coverage': {'title': 'Coverage', 'options': ['FD']},
 'repeatCycleIdentifier': {'title': 'Repeat Cycle', 'options': []},
 'centerOfLongitude': {'title': 'Sub Sat Longitude', 'options': ['0.0']},
 'sort': {'title': 'SRU sort keys, see OpenSearch Extension for EO f

In [8]:
latest = selected_collection.search().first()

try:
    print(latest)
except eumdac.collection.CollectionError as error:
    print(f"Error related to the collection: '{error.msg}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260930151237_IDPFI_OPE_20260930151006_20260930151923_N__O_0092_0000


In [9]:
satellite_type = "Metop-C"
products = selected_collection.search(sat=satellite_type)

try:
    print(f'Found Datasets: {products.total_results} datasets for the given satellite type.')
except eumdac.collection.CollectionError as error:
    print(f"Error related to the collection: '{error.msg}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

Found Datasets: 0 datasets for the given satellite type.


In [10]:
# Set bounding-box coordinates
area = '6.0, 36.5, 19.0, 47.5'  # lon_min, lat_min, lon_max, lat_max# Retrieve datasets that match our filter
products = selected_collection.search(bbox=area)

try:
    print(f'Found Datasets: {products.total_results} datasets for the given area of interest')
except eumdac.collection.CollectionError as error:
    print(f"Error related to the collection: '{error.msg}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

Found Datasets: 103453 datasets for the given area of interest


In [11]:
# Set sensing start and end time
start = datetime.datetime(2026, 7, 20, 12, 0)
end = datetime.datetime(2026, 7, 20, 18, 0)
# Retrieve datasets that match our filter
products = selected_collection.search(
    dtstart=start,
    dtend=end,
    sort="start,time,1")

try:
    print(f'Found {products.total_results} datasets for the given time range and geometry:')
    for product in products:
        print(product)
except eumdac.collection.CollectionError as error:
    print(f"Error related to the collection: '{error.msg}'")
except requests.exceptions.RequestException as error:
    print(f"Unexpected error: {error}")

Found 36 datasets for the given time range and geometry:
W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260720120324_IDPFI_OPE_20260720120007_20260720120939_N__O_0073_0000
W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260720121337_IDPFI_OPE_20260720121007_20260720121939_N__O_0074_0000
W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260720122316_IDPFI_OPE_20260720122007_20260720122939_N__O_0075_0000
W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260720123310_IDPFI_OPE_20260720123007_20260720123939_N__O_0076_0000
W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260720124312_IDPFI_OPE_20260720124007_20260720124939_N__O_0077_0000
W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260720125308_IDPFI_OPE_20260720125007_20260720125939_N__O_0078_0000
W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260

# Downloading

In [12]:
for product in products:
    print(product)
    try:
        with product.open() as fsrc, \
                open(fsrc.name, mode='wb') as fdst:
            shutil.copyfileobj(fsrc, fdst)
            print(f'Download of product {product} finished.')
    except eumdac.product.ProductError as error:
        print(f"Error related to the product '{product}' while trying to download it: '{error.msg}'")
    except requests.exceptions.ConnectionError as error:
        print(f"Error related to the connection: '{error.msg}'")
    except requests.exceptions.RequestException as error:
        print(f"Unexpected error: {error}")

print('All downloads are finished.')

W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20260720120324_IDPFI_OPE_20260720120007_20260720120939_N__O_0073_0000


AttributeError: 'ProductError' object has no attribute 'msg'

In [13]:
import eumdac

token = eumdac.AccessToken((consumer_key, consumer_secret))
print(token.expiration)                    # should print a future time
datastore = eumdac.DataStore(token)

# 1. Can you see other collections and search them?
for cid in ["EO:EUM:DAT:0662", "EO:EUM:DAT:0686", "EO:EUM:DAT:MSG:HRSEVIRI"]:
    try:
        col = datastore.get_collection(cid)
        prods = col.search(dtstart=datetime.datetime(2025, 7, 15, 12, 0),
                           dtend=datetime.datetime(2025, 7, 15, 12, 30))
        print(cid, "search OK:", prods.total_results)
        p = prods.first()
        with p.open(entry=p.entries[0]) as f:   # tiny download test
            f.read(1024)
        print(cid, "download OK")
    except Exception as e:
        print(cid, "FAILED:", type(e).__name__, e)

2026-10-01 12:01:24.589629
EO:EUM:DAT:0662 search OK: 3
EO:EUM:DAT:0662 FAILED: ProductError Could not download Product W_XX-EUMETSAT-Darmstadt,IMG+SAT,MTI1+FCI-1C-RRAD-FDHSI-FD--x-x---x_C_EUMT_20250715122329_IDPFI_OPE_20250715122007_20250715122934_N__O_0075_0000 of Collection EO:EUM:DAT:0662 - Unauthorised (403)
EO:EUM:DAT:0686 search OK: 5
EO:EUM:DAT:0686 download OK
EO:EUM:DAT:MSG:HRSEVIRI search OK: 2
EO:EUM:DAT:MSG:HRSEVIRI FAILED: ProductError Could not download Product MSG3-SEVI-MSG15-0100-NA-20250715122743.318000000Z-NA of Collection EO:EUM:DAT:MSG:HRSEVIRI - Unauthorised (403)
